In [ ]:
#196法1
import pandas as pd
# 示例数据
data = [
    [1, 'john@example.com'],
    [2, 'bob@example.com'],
    [3, 'john@example.com']
]
person = pd.DataFrame(data, columns=['id', 'email'])
# 先按 id 升序排序，确保最小 id 在前
person.sort_values('id',inplace=True)
# 删除重复 email，只保留最小 id
person.drop_duplicates(subset='email', keep='first',inplace=True)
person


,id,email
0,1,john@example.com
1,2,bob@example.com


In [ ]:
#196 fa2
data = [
    [1, 'john@example.com'],
    [2, 'bob@example.com'],
    [3, 'john@example.com']
]
person = pd.DataFrame(data, columns=['id', 'email'])
min_id = person.groupby('email')['id'].transform('min')
removed_person = person[person['id'] != min_id]
person.drop(removed_person.index, inplace=True)
person


,id,email
0,1,john@example.com
1,2,bob@example.com


In [31]:
#511
import pandas as pd
# 假设 Activity 表数据如下
data = [
    [1, 2, '2016-03-01', 5],
    [1, 2, '2016-05-02', 6],
    [2, 3, '2017-06-25', 1],
    [3, 1, '2016-03-02', 0],
    [3, 4, '2018-07-03', 5]
]
# 创建 DataFrame
df = pd.DataFrame(data, columns=['player_id', 'device_id',\
     'event_date', 'games_played'])
# 确保 event_date 是 datetime 类型
df['event_date'] = pd.to_datetime(df['event_date'])
# 按 player_id 分组取最早的 event_date
result = df.groupby('player_id')['event_date'].min().reset_index()
# 重命名列
result = result.rename(columns={'event_date': 'first_login'})
print(result)


   player_id first_login
0          1  2016-03-01
1          2  2017-06-25
2          3  2016-03-02


In [22]:
#570
import pandas as pd
# 示例数据
data = {
    'id': [101, 102, 103, 104, 105, 106],
    'name': ['John', 'Dan', 'James', 'Amy', 'Anne', 'Ron'],
    'department': ['A', 'A', 'A', 'A', 'A', 'B'],
    'managerId': [None, 101, 101, 101, 101, 101]
}
df = pd.DataFrame(data)
# 统计每个 managerId 的直接下属数量
sub_counts = df.groupby('managerId').size().reset_index(name='sub_count')
sub_counts
# 筛选下属数 >= 5 的 managerId
managers_with_5_plus = sub_counts[sub_counts['sub_count'] >= 5]
managers_with_5_plus 
# 获取经理名字
result = df[df['id'].isin(managers_with_5_plus['managerId'])][['name']]
print(result)


,managerId,sub_count
0,101.0,5


,managerId,sub_count
0,101.0,5


   name
0  John


In [28]:
import pandas as pd
import re
# 读取原始数据
df = pd.read_csv('../data/豆瓣.csv')
df.dtypes

电影中文名     object
电影外文名     object
其他信息      object
评分       float64
评分人数      object
简介        object
dtype: object

(250, 6)

In [10]:
import pandas as pd
import re
# 读取原始数据
df = pd.read_csv('../data/豆瓣.csv')
# 数据清洗和处理函数
def safe_strip(text):
    """安全的strip函数，处理空值"""
    if pd.isna(text):
        return ''
    return str(text).strip()
def extract_director_chinese(info):
    """提取导演中文名"""
    if pd.isna(info):
        return ''
    match = re.search(r'导演:\s*([^\(\n]+?)\s+[A-Z]', str(info))
    return match.group(1).strip() if match else ''
def extract_director_foreign(info):
    """提取导演外文名"""
    if pd.isna(info):
        return ''
    match = re.search(r'导演:[^\(]*\(([^\)]+)\)', str(info))
    return match.group(1).strip() if match else ''
def extract_actor_chinese(info):
    """提取主演中文名"""
    if pd.isna(info):
        return ''
    match = re.search(r'主演:\s*([^\(\n]+?)\s+[A-Z]', str(info))
    if match:
        # 提取第一个主演的中文名
        actors = match.group(1).strip().split('/')[0].strip()
        return actors
    return ''
def extract_actor_foreign(info):
    """提取主演外文名"""
    if pd.isna(info):
        return ''
    match = re.search(r'主演:[^\(]*\(([^\)]+)\)', str(info))
    if match:
        # 提取第一个主演的外文名
        actors = match.group(1).strip().split('/')[0].strip()
        return actors
    return ''
def extract_year(info):
    """提取上映年份"""
    if pd.isna(info):
        return ''
    match = re.search(r'(\d{4})', str(info).split('/')[0])
    return match.group(1) if match else ''
def extract_country(info):
    """提取国家"""
    if pd.isna(info):
        return ''
    parts = str(info).split('/')
    if len(parts) > 1:
        return parts[1].strip()
    return ''
def extract_genre(info):
    """提取电影类型"""
    if pd.isna(info):
        return ''
    parts = str(info).split('/')
    if len(parts) > 2:
        return parts[2].strip()
    return ''
def clean_foreign_name(foreign_name):
    """清洗外文名"""
    if pd.isna(foreign_name):
        return ''
    # 移除开头的斜杠和空格
    cleaned = re.sub(r'^\s*\/\s*', '', str(foreign_name))
    # 如果有多个外文名，取第一个
    if ' / ' in cleaned:
        cleaned = cleaned.split(' / ')[0]
    return cleaned.strip()
def clean_rating_count(rating_count):
    """清洗评分人数"""
    if pd.isna(rating_count):
        return ''
    return re.sub(r'人评价$', '', str(rating_count))
# 处理数据
processed_data = []
for _, row in df.iterrows():
    processed_row = {
        '电影中文名': safe_strip(row['电影中文名']),
        '电影外文名': clean_foreign_name(row['电影外文名']),
        '导演中文名': extract_director_chinese(row['其他信息']),
        '导演外文名': extract_director_foreign(row['其他信息']),
        '主演中文名': extract_actor_chinese(row['其他信息']),
        '主演外文名': extract_actor_foreign(row['其他信息']),
        '评分': row['评分'],
        '评分人数': clean_rating_count(row['评分人数']),
        '简介': safe_strip(row['简介']),
        '上映年份': extract_year(row['其他信息']),
        '国家': extract_country(row['其他信息']),
        '电影类型': extract_genre(row['其他信息'])
    }
    processed_data.append(processed_row)
# 创建新的DataFrame
processed_df = pd.DataFrame(processed_data)
# 显示前几行数据
print("处理后的数据前10行：")
print(processed_df.head(10))
# 显示数据基本信息
print("\n数据基本信息：")
print(f"总行数: {len(processed_df)}")
print(f"列名: {list(processed_df.columns)}")
# 检查是否有空值
print("\n空值统计：")
print(processed_df.isnull().sum())
# 保存处理后的数据
processed_df.to_csv('豆瓣_处理后.csv', index=False, encoding='utf-8-sig')
print("\n数据已保存到 '豆瓣_处理后.csv'")
# 显示一些示例数据
print("\n示例数据（前5行）：")
for i in range(min(5, len(processed_df))):
    print(f"\n电影 {i+1}:")
    print(f"  中文名: {processed_df.iloc[i]['电影中文名']}")
    print(f"  外文名: {processed_df.iloc[i]['电影外文名']}")
    print(f"  导演: {processed_df.iloc[i]['导演中文名']} ({processed_df.iloc[i]['导演外文名']})")
    print(f"  主演: {processed_df.iloc[i]['主演中文名']} ({processed_df.iloc[i]['主演外文名']})")
    print(f"  年份: {processed_df.iloc[i]['上映年份']}")
    print(f"  国家: {processed_df.iloc[i]['国家']}")
    print(f"  类型: {processed_df.iloc[i]['电影类型']}")

处理后的数据前10行：
     电影中文名                     电影外文名     导演中文名 导演外文名       主演中文名 主演外文名   评分  \
0   肖申克的救赎  The Shawshank Redemption  弗兰克·德拉邦特            蒂姆·罗宾斯        9.7   
1     霸王别姬                    再见，我的妾       陈凯歌               张国荣        9.6   
2     阿甘正传              Forrest Gump  罗伯特·泽米吉斯            汤姆·汉克斯        9.5   
3    泰坦尼克号                   Titanic   詹姆斯·卡梅隆        莱昂纳多·迪卡普里奥        9.5   
4     千与千寻                  千と千尋の神隠し       宫崎骏               柊瑠美        9.4   
5  这个杀手不太冷                      Léon     吕克·贝松              让·雷诺        9.4   
6     美丽人生           La vita è bella   罗伯托·贝尼尼           罗伯托·贝尼尼        9.5   
7     星际穿越              Interstellar  克里斯托弗·诺兰            马修·麦康纳        9.4   
8     盗梦空间                 Inception  克里斯托弗·诺兰        莱昂纳多·迪卡普里奥        9.4   
9    楚门的世界           The Truman Show     彼得·威尔              金·凯瑞        9.4   

      评分人数                     简介  上映年份  \
0  3034225                希望让人自由。         
1  2242764                  风华绝代

In [26]:
processed_df.to_csv('豆瓣_处理后.csv', index=False, encoding='utf-8-sig')
print(f"\n数据已保存到 '豆瓣_处理后.csv'，共{len(processed_df)}条记录")


数据已保存到 '豆瓣_处理后.csv'，共250条记录
